# Colab: SFT-only training (minimal)

Stripped to the bone. Clone repo, install peft + trl, run SFT. No eval, no GRPO, no bitsandbytes, no torchvision workarounds. Runtime: T4 (bf16, no 4-bit quant).

In [ ]:
# 1. Clone repo (idempotent).
import os
if os.path.isdir('ml-project'):
    !cd ml-project && git pull
else:
    !git clone https://github.com/Andrii238/ml-project.git
%cd ml-project

In [ ]:
# 2. Remove Colab's two broken preinstalled packages before any import.
# bitsandbytes: Colab ships a mixed-version native lib that segfaults on import.
# torchvision: torch/torchvision ABI mismatch on Colab — crashes at C level.
# We use neither: Qwen 1.5B fits on T4 in bf16, no vision models.
# Doing this BEFORE the first import means no runtime restart needed.
!pip uninstall -y -q bitsandbytes torchvision
# Install only peft + trl on top of Colab's working stack.
!pip install -q -r requirements-colab.txt

In [ ]:
# 3. Sanity check — GPU + imports.
import torch
print('cuda:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no gpu')
import transformers, trl, peft
print('transformers', transformers.__version__, '| trl', trl.__version__, '| peft', peft.__version__)

In [ ]:
# 4. Run SFT training. Saves LoRA adapter to ckpts/sft.
from training.train_sft import train as sft_train, SFTConfig

sft_train(SFTConfig(
    output_dir='ckpts/sft',
    epochs=3,
    per_device_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    load_in_4bit=False,
))